# CS 3110/5110: Data Privacy
## In-Class Exercise, week of 11/2/2026

In [1]:
# Load the data and libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)

def gaussian_mech(v, sensitivity, epsilon, delta):
    return v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)

def gaussian_mech_vec(vec, sensitivity, epsilon, delta):
    return [v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)
            for v in vec]

def pct_error(orig, priv):
    return np.abs(orig - priv)/orig * 100.0

# adult = pd.read_csv('https://github.com/jnear/cs211-data-privacy/raw/master/homework/adult_with_pii.csv')

In [2]:
# Load data files
import numpy as np
import urllib.request
import io

url_x = 'https://github.com/jnear/cs211-data-privacy/raw/master/slides/adult_processed_x.npy'
url_y = 'https://github.com/jnear/cs211-data-privacy/raw/master/slides/adult_processed_y.npy'

with urllib.request.urlopen(url_x) as url:
    f = io.BytesIO(url.read())
X = np.load(f)

with urllib.request.urlopen(url_y) as url:
    f = io.BytesIO(url.read())
y = np.load(f)

In [3]:
# Split data into training and test sets
training_size = int(X.shape[0] * 0.8)

X_train = X[:training_size]
X_test = X[training_size:]

y_train = y[:training_size]
y_test = y[training_size:]

print('Train and test set sizes:', len(y_train), len(y_test))

Train and test set sizes: 36176 9044


## Question 1

Using scikit-learn, train a logistic regression model on the training data loaded above.

In [4]:
from sklearn.linear_model import LogisticRegression

In [5]:
def train_model():
    model = LogisticRegression().fit(X_train, y_train)
    return model

model = train_model()
print('Model coefficients:', model.coef_[0])
print('Model accuracy:', np.sum(model.predict(X_test) == y_test)/X_test.shape[0])

Model coefficients: [ 3.22958882e-01 -3.57306784e-01 -1.85450982e-01 -8.06914609e-02
 -7.39295462e-01 -5.27268957e-01 -9.29960592e-01 -7.74714555e-01
 -7.32356354e-01 -5.30503787e-01 -1.13074465e+00 -5.65991641e-01
 -8.59361273e-01 -8.89811164e-01  5.13122916e-02  8.49360041e-02
  5.13313049e-01  1.04764160e+00 -1.76361460e-01  7.64755656e-01
 -6.51767514e-01  1.29011170e+00  6.25267372e-02 -1.02809816e+00
  1.35034986e+00  1.22888768e+00 -7.10258159e-01 -1.43669447e+00
 -1.01748435e+00 -8.83717750e-01 -8.37513280e-02 -1.03160599e-02
 -3.21459143e-02  6.88679380e-01 -9.40116076e-01 -7.35491695e-01
 -3.63520836e-01 -9.45370642e-01 -1.45310889e+00  4.44083102e-01
  4.60628173e-01  1.87508864e-01  4.61468437e-01 -1.75561874e-01
 -5.68973781e-01 -8.68334975e-02 -9.23221803e-01 -1.15708883e+00
 -2.76079894e-01  5.15182447e-01 -6.78048800e-01 -1.74817687e-01
 -6.68065266e-01 -5.67879787e-01 -4.08203816e-01 -1.58428676e+00
 -9.12728599e-01  8.58290126e-01  6.67071807e-01 -3.23352899e-01
 -1.3

## Question 2

Implement the *average gradient* of the loss below.

In [6]:
# The loss function measures how good our model is. The training goal is to minimize the loss.
# This is the logistic loss function.
def loss(theta, xi, yi):
    exponent = - yi * (xi.dot(theta))
    return np.log(1 + np.exp(exponent))

# This is the gradient of the logistic loss
# The gradient is a vector that indicates the rate of change of the loss in each direction
def gradient(theta, xi, yi):
    exponent = yi * (xi.dot(theta))
    return - (yi*xi) / (1+np.exp(exponent))

In [7]:
theta = np.zeros(104) # Bad model
theta = model.coef_[0] # This is the sklearn model, much better
loss(theta, X_train[2], y_train[2])

np.float64(0.004126559397270145)

In [8]:
def avg_grad(theta, X, y):
    all_grads = [gradient(theta, xi, yi) for xi, yi, in zip(X, y)]
    return np.mean(all_grads, axis=0)

In [9]:
avg_grad(theta, X_train, y_train)

array([1.36482039e-02, 3.10788626e-02, 3.09874461e-01, 1.30246053e-02,
       4.08604819e-02, 1.89191026e-02, 2.51008290e-04, 9.38017691e-03,
       1.08935141e-02, 4.24666262e-03, 1.39778520e-03, 3.50008024e-03,
       7.66854895e-03, 5.43937421e-03, 1.55694365e-02, 2.13615064e-02,
       7.25216930e-02, 3.05467484e-03, 1.52972397e-01, 2.07780775e-02,
       4.60351560e-04, 3.74530642e-03, 9.46671393e-02, 6.98475016e-02,
       2.27885478e-04, 2.18833981e-01, 5.21538613e-03, 1.07218752e-01,
       1.29866593e-02, 1.33265600e-02, 5.23964843e-02, 1.33009543e-04,
       7.08503859e-02, 5.42689968e-02, 1.44879028e-02, 1.59576122e-02,
       3.05835954e-02, 2.96265941e-02, 8.47397119e-04, 5.46291098e-02,
       1.06315754e-02, 5.09202577e-02, 1.50292507e-02, 2.72945536e-02,
       1.94172527e-01, 1.29919717e-01, 8.38233085e-03, 2.96282588e-02,
       4.48331888e-02, 2.07207032e-02, 4.14011004e-03, 1.20634185e-02,
       3.63241463e-02, 3.01568299e-03, 3.72113368e-01, 1.25378303e-01,
      

## Question 3

Use the average gradient from above to implement a gradient descent algorithm.

In [10]:
def gradient_descent(iterations):
    # Start with mad model
    theta = np.zeros(104)
    # For iterations times
    for _ in range(iterations):
        # Compute gradients
        grad = avg_grad(theta, X_train, y_train)
        # Update model
        theta = theta - grad
    return theta



theta = gradient_descent(20)
theta

array([ 3.21901256e-02, -5.07200485e-02, -4.34345416e-01,  8.96964711e-02,
       -1.37010803e-01, -5.11151234e-02, -2.44916217e-03, -1.03007575e-01,
       -1.27977275e-01, -4.44061464e-02, -2.99109248e-02, -5.33534238e-02,
       -8.94049056e-02, -6.87079251e-02, -1.17712378e-02, -3.03750555e-02,
        2.65800195e-01,  8.22493467e-02, -4.64614683e-01,  2.13360579e-01,
       -8.66724247e-03,  1.24924289e-01, -2.07891971e-01, -2.34116871e-01,
        3.33385154e-03,  5.36454877e-01, -2.94018526e-02, -6.91538796e-01,
       -8.23694335e-02, -5.61157308e-02, -1.62569054e-01, -2.72058745e-04,
       -1.74412543e-01,  3.29412970e-01, -1.21355180e-01, -1.42347161e-01,
       -1.63259663e-01, -3.12545311e-01, -1.71753564e-02,  2.85631912e-01,
       -5.70876128e-03,  5.11511640e-03,  1.39955040e-02, -8.82643699e-02,
        3.95160622e-01, -3.56942896e-01, -9.65065203e-02, -3.99816274e-01,
       -2.57935026e-01,  1.62286138e-01, -2.69748982e-02, -4.33001351e-02,
       -2.04640240e-01, -

In [11]:
# Prediction: take a model (theta) and a single example (xi) and return its predicted label
def predict(xi, theta, bias=0):
    label = np.sign(xi @ theta + bias)
    return label

def accuracy(theta):
    return np.sum(predict(X_test, theta) == y_test)/X_test.shape[0]

accuracy(theta)

np.float64(0.8081601061477223)

## Question 4

Implement a *noisy gradient descent* algorithm.

1. Calculate gradients for each example
2. Clip the gradients to have bounded $L2$ norm
3. Sum the clipped gradients
4. Use the Gaussian mechanism to add noise to the sum of gradients

In [14]:
def L2_clip(v, b):
    norm = np.linalg.norm(v, ord=2)
    
    if norm > b:
        return b * (v / norm)
    else:
        return v

def noisy_gradient_descent(iterations, epsilon, delta):
    # Start with mad model
    theta = np.zeros(104)
    # For iterations times
    b= 3
    noisy_count = laplace_mech(len(X_train), epsilon=epsilon , sensitivity=1)
    for _ in range(iterations):
        # Compute gradients
        # grad = avg_grad(theta, X_train, y_train)
        all_grads = [gradient(theta, x_i, y_i) for x_i, y_i in zip(X_train, y_train)]

        clipped_grads = [L2_clip(g, b) for g in all_grads]
        clipped_grad_sum = np.sum(clipped_grads, axis=0)

        noisy_grad_sum = gaussian_mech_vec(clipped_grad_sum, sensitivity=b, epsilon=epsilon, delta=delta)
        avg_grad = np.array(noisy_grad_sum) / len(X_train)

        # Update model
        theta = theta - avg_grad
    return theta


theta = noisy_gradient_descent(10, 1.0, 1e-5)
print('Final accuracy:', accuracy(theta))

Final accuracy: 0.7784166298098186


In [ ]:
# TEST CASE

assert accuracy(noisy_gradient_descent(5, 0.001, 1e-5)) < 0.76
assert accuracy(noisy_gradient_descent(5, 1.0, 1e-5)) > 0.70

## Question 5

What is the *total privacy cost* of the noisy gradient descent algorithm above, and why? Argue informally that the algorithm satisfies this privacy cost. Use sequential composition.

* Sens
    noisy_count: counting query, sens is 1

    noisy_grad_sum: We sum a bunch of gradient vectors. Each gradient vector is clipped to have L2 Norm bounded by b, Therefore the Sum has L2 sens bounded by b.

* Composition
    We used the laplace mech vech once with parameter epsilon, we used the gaussian mech vec iterations times with parameters and delta. The total privacy cost is epsilon + iterations * epislon, iterations * delta by seq comp

* Post Proc

We return theta. THeta starts out as a constant (all zero), and everytime we update it we add a noisy gradient that satisfies DP. Therefore theta always satisfies DP by post proc

## Question 6

Repeat the above, but using advanced composition.

Apply the equation from textbook

## Question 7

Implement a version of noisy gradient descent that satisfies a *total* of $(\epsilon, \delta)$-differential privacy. Use sequential composition.

In [ ]:
def noisy_gradient_descent(iterations, epsilon, delta):
    # Start with mad model
    theta = np.zeros(104)
    # For iterations times
    b= 3
    epsilon_i = epsilon/(iterations + 1)
    delta_i = delta/iterations
    noisy_count = laplace_mech(len(X_train), epsilon=epsilon_i , sensitivity=1)
    for _ in range(iterations):
        # Compute gradients
        # grad = avg_grad(theta, X_train, y_train)
        all_grads = [gradient(theta, x_i, y_i) for x_i, y_i in zip(X_train.values, y_train)]
        clipped_grad_sum = [L2_clip(g, b) for g in all_grads]
        noisy_grad_sum = gaussian_mech_vec(clipped_grad_sum, sensitivity=b, epsilon=epsilon_i, delta=delta_i)
        avg_grad = np.array(noisy_grad_sum) / noisy_count

        # Update model
        theta = theta - avg_grad
    return theta


theta = noisy_gradient_descent(10, 1.0, 1e-5)
print('Final accuracy:', accuracy(theta))

# To convert to zCDP: Just use gaussian_mech_zcdp_vec and rho instead of epsilon
# Instead of laplace mech for noisy_count? use gaussian mech_zcdp for it instead of laplace

NotImplementedError: 

In [ ]:
# TEST CASE

assert accuracy(noisy_gradient_descent(5, 0.001, 1e-5)) < 0.76
assert accuracy(noisy_gradient_descent(5, 1.0, 1e-5)) > 0.70